# Evaluation

Tahap evaluasi (*Evaluation*) bertugas memuat model-model yang telah dilatih dari tahap *Modeling*, mengevaluasi performa klasterisasi berdasarkan metrik internal, memilih algoritma terbaik, serta menghasilkan dataset kabupaten/kota terklaster (`clustered_regencies.csv`).


In [ ]:
import os

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from config import (
    PREPARED_REGENCIES_CSV,
    FEATURE_SELECTION_JSON,
    KMEANS_MODEL_PKL,
    AGGLOMERATIVE_MODEL_PKL,
    CLUSTERED_REGENCIES_CSV,
    MODEL_COMPARISON_JSON
)


## Pemuatan Data & Model Terlatih

In [ ]:
df_reg = pd.read_csv(PREPARED_REGENCIES_CSV)
feat_config = json.load(open(FEATURE_SELECTION_JSON, 'r', encoding='utf-8'))
scaled_cols = feat_config.get('scaled_feature_columns', [c for c in df_reg.columns if c.startswith('scaled_')])
X = df_reg[scaled_cols].values

# Load trained models from artifact/3_modeling/
kmeans_model = pickle.load(open(KMEANS_MODEL_PKL, 'rb'))
agg_model = pickle.load(open(AGGLOMERATIVE_MODEL_PKL, 'rb'))

trained_models = {
    'K-Means': kmeans_model,
    'Agglomerative': agg_model
}

print(f'Total Baris Data        : {len(df_reg)}')
print(f'Kolom Fitur Cluster     : {scaled_cols}')
print(f'Model Berhasil Dimuat   : {list(trained_models.keys())}')


## Evaluasi & Komparasi Model

In [ ]:
comparison_records = []

for name, model in trained_models.items():
    labels = model.labels_
    sil = round(float(silhouette_score(X, labels)), 4)
    ch = round(float(calinski_harabasz_score(X, labels)), 2)
    db = round(float(davies_bouldin_score(X, labels)), 4)

    comparison_records.append({
        'Algoritma': name,
        'Jumlah Klaster': len(np.unique(labels)),
        'Silhouette': sil,
        'Calinski-Harabasz': ch,
        'Davies-Bouldin': db
    })

df_comparison = pd.DataFrame(comparison_records)
df_sorted = df_comparison.sort_values(by=['Silhouette', 'Calinski-Harabasz'], ascending=[False, False]).reset_index(drop=True)

print("Tabel Perbandingan Performa Algoritma Klasterisasi:")
print(df_sorted.to_markdown(index=False))


## Pemilihan Model Terbaik & Penyimpanan Hasil Klasterisasi

In [ ]:
best_row = df_sorted.iloc[0]
best_algo = best_row['Algoritma']
best_model = trained_models[best_algo]
best_k = int(best_row['Jumlah Klaster'])

# 1. Tetapkan label klaster dan simpan seluruh kolom dataset prepared + cluster_label
df_clustered = df_reg.copy()
df_clustered['cluster_label'] = best_model.labels_.astype(int)

int_cols = ['province_id', 'regency_no', 'total_koperasi', 'koperasi_nib', 'koperasi_npwp', 'koperasi_rat', 'cluster_label']
for col in int_cols:
  if col in df_clustered.columns:
    df_clustered[col] = df_clustered[col].round().astype(int)

os.makedirs(os.path.dirname(CLUSTERED_REGENCIES_CSV), exist_ok=True)
df_clustered.to_csv(CLUSTERED_REGENCIES_CSV, index=False)
print(f'Model terbaik terpilih: {best_algo} (K={best_k})')
print(f'Dataset hasil klasterisasi tersimpan di: {CLUSTERED_REGENCIES_CSV}')

# 2. Simpan ringkasan evaluasi komparasi ke JSON
comparison_summary = {
    'best_algorithm': best_algo,
    'best_k': best_k,
    'comparison_table': df_sorted.to_dict(orient='records')
}

json.dump(comparison_summary, open(MODEL_COMPARISON_JSON, 'w', encoding='utf-8'), indent=2)
print(f'Ringkasan evaluasi perbandingan tersimpan di: {MODEL_COMPARISON_JSON}')


## Metrik Validasi Internal Klaster Terbaik

In [ ]:
sil_score = best_row['Silhouette']
ch_score = best_row['Calinski-Harabasz']
db_score = best_row['Davies-Bouldin']

eval_df = pd.DataFrame({
    'Metrik Validasi': [
        'Model Algoritma',
        'Jumlah Klaster (K)',
        'Silhouette Coefficient',
        'Calinski-Harabasz Index',
        'Davies-Bouldin Index'
    ],
    'Nilai Evaluasi': [
        f"{best_algo} (Terbaik)",
        best_k,
        sil_score,
        ch_score,
        db_score
    ],
    'Kriteria Evaluasi': [
        'Best Selected',
        'Optimal K',
        'Semakin Mendekati +1 Semakin Baik',
        'Semakin Tinggi Semakin Baik',
        'Semakin Rendah Semakin Baik'
    ]
})
print(eval_df.to_markdown(index=False))


## Eksplorasi Visual Hasil Klasterisasi

### Proyeksi 2D Klaster (PCA)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
df_pca = pd.DataFrame(X_pca, columns=['PCA1', 'PCA2'])
df_pca['cluster'] = [f"Klaster {lbl}" for lbl in df_clustered['cluster_label'].values]

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='cluster', data=df_pca, palette='tab10', s=60, alpha=0.85)
plt.title(f'Proyeksi 2D Klasterisasi ({best_algo}, K={best_k}) - PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Klaster')
plt.tight_layout()
plt.show()


### Distribusi Jumlah Anggota per Klaster

In [ ]:
cluster_dist = df_clustered['cluster_label'].value_counts().sort_index()

klaster_names = [f"Klaster {c}" for c in cluster_dist.index]
persentase_values = [round((count / len(df_clustered)) * 100, 2) for count in cluster_dist.values]

dist_df = pd.DataFrame({
    'Klaster': klaster_names,
    'Jumlah Kab/Kota': cluster_dist.values,
    'Persentase (%)': persentase_values
})

print(dist_df.to_markdown(index=False))

plt.figure(figsize=(7, 4))
sns.barplot(x='Klaster', y='Jumlah Kab/Kota', data=dist_df, palette='viridis')
plt.title('Distribusi Anggota per Klaster')
plt.ylabel('Jumlah Kabupaten / Kota')
plt.tight_layout()
plt.show()
